# 🇨🇳 Análise dos Empréstimos Chineses — GCDF + BRI + Colaterais
Datasets: AidData GCDF v2.0 | BRI Tagged v1.0 | How China Collateralizes v1.0

## 1. Carregamento e União dos Datasets

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Configurações visuais
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11

# Carregando os datasets
gcdf = pd.read_excel("AidDatasGlobalChineseDevelopmentFinanceDataset_v2_0.xlsx", sheet_name="Global_CDF2.0")
bri  = pd.read_excel("BRI_Tagged_Chinese_Official_Finance_Dataset_v1.xlsx", sheet_name="Sheet 1")
col  = pd.read_excel("How_China_Collateralizes_Dataset_Version_1_0.xlsx", sheet_name="Data")

# Colunas BRI a adicionar
colunas_bri = [
    'AidData TUFF Project ID',
    'BRI_1 Policy Communication',
    'BRI_2 Infrastructure Connectivity',
    'BRI_3 Trade',
    'BRI_4 Monetary Circulation',
    'BRI_5 People to People',
    'BRI-Like', 'Non-BRI', 'Vague-BRI'
]

# União dos datasets
df = pd.merge(gcdf, bri[colunas_bri], on='AidData TUFF Project ID', how='left')
col_renamed = col.rename(columns={'AidData Record ID': 'AidData TUFF Project ID'})
df = pd.merge(df, col_renamed, on='AidData TUFF Project ID', how='left')

# Renomear colunas duplicadas para facilitar
df = df.rename(columns={
    'Recipient_x': 'País',
    'Recipient Region_x': 'Região',
    'Commitment Year_x': 'Ano',
    'Sector Name': 'Setor',
    'Amount (Constant USD2017)': 'Valor_USD'
})

print(f'✅ Dataset unido: {df.shape[0]} projetos, {df.shape[1]} colunas')
df[['País', 'Região', 'Ano', 'Setor', 'Valor_USD', 'BRI-Like', 'Non-BRI']].head()

---
## 2. Projetos BRI por Região

In [ ]:
# Filtrar apenas projetos BRI-Like
bri_df = df[df['BRI-Like'] == 1].copy()

# Contagem de projetos BRI por região
projetos_por_regiao = bri_df.groupby('Região')['AidData TUFF Project ID'].count().sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico 1: Número de projetos
projetos_por_regiao.plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Número de Projetos BRI por Região')
axes[0].set_xlabel('Quantidade de Projetos')
axes[0].set_ylabel('')

# Gráfico 2: Valor total por região
valor_por_regiao = (bri_df.groupby('Região')['Valor_USD'].sum() / 1e9).sort_values(ascending=True)
valor_por_regiao.plot(kind='barh', ax=axes[1], color='darkorange')
axes[1].set_title('Valor Total de Projetos BRI por Região (USD 2017)')
axes[1].set_xlabel('Bilhões de USD')
axes[1].set_ylabel('')

plt.suptitle('Projetos BRI por Região', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Top 15 países com mais projetos BRI
top_paises = bri_df.groupby('País')['AidData TUFF Project ID'].count().sort_values(ascending=False).head(15)

top_paises.sort_values().plot(kind='barh', color='steelblue')
plt.title('Top 15 Países com Mais Projetos BRI')
plt.xlabel('Quantidade de Projetos')
plt.tight_layout()
plt.show()

---
## 3. Setores Mais Financiados

In [ ]:
# Top 10 setores por valor total (todos os projetos)
setores_valor = (
    df.groupby('Setor')['Valor_USD']
    .sum()
    .dropna()
    .sort_values(ascending=False)
    .head(10) / 1e9
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Por valor
setores_valor.sort_values().plot(kind='barh', ax=axes[0], color='seagreen')
axes[0].set_title('Top 10 Setores por Valor Total (Bilhões USD)')
axes[0].set_xlabel('Bilhões de USD (constante 2017)')
axes[0].set_ylabel('')

# Por número de projetos
setores_count = df['Setor'].value_counts().head(10).sort_values()
setores_count.plot(kind='barh', ax=axes[1], color='mediumpurple')
axes[1].set_title('Top 10 Setores por Número de Projetos')
axes[1].set_xlabel('Quantidade de Projetos')
axes[1].set_ylabel('')

plt.suptitle('Setores Mais Financiados pela China', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Comparação: setores BRI vs Não-BRI
bri_setores = bri_df['Setor'].value_counts().head(8)
nao_bri_setores = df[df['Non-BRI'] == 1]['Setor'].value_counts().head(8)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bri_setores.sort_values().plot(kind='barh', ax=axes[0], color='darkorange')
axes[0].set_title('Top Setores — Projetos BRI')
axes[0].set_xlabel('Quantidade de Projetos')

nao_bri_setores.sort_values().plot(kind='barh', ax=axes[1], color='steelblue')
axes[1].set_title('Top Setores — Projetos Não-BRI')
axes[1].set_xlabel('Quantidade de Projetos')

plt.suptitle('Setores: BRI vs Não-BRI', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 4. Colaterais e Garantias

In [ ]:
# Proporção de empréstimos com colateral
col_df = df[df['Collateralized/Securitized'].notna()].copy()
col_counts = col_df['Collateralized/Securitized'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Pizza
total_com = col_counts.get('Yes', 0)
total_sem = df.shape[0] - total_com
axes[0].pie(
    [total_com, total_sem],
    labels=['Com Colateral', 'Sem Colateral'],
    autopct='%1.1f%%',
    colors=['tomato', 'lightgray'],
    startangle=90
)
axes[0].set_title('Proporção de Projetos com Colateral')

# Tipo de commodity usada como colateral
commodities = df['Commodity'].value_counts().dropna().head(8).sort_values()
if len(commodities) > 0:
    commodities.plot(kind='barh', ax=axes[1], color='tomato')
    axes[1].set_title('Commodities Usadas como Colateral')
    axes[1].set_xlabel('Quantidade de Projetos')
else:
    axes[1].text(0.5, 0.5, 'Dados insuficientes', ha='center', va='center')
    axes[1].set_title('Commodities Usadas como Colateral')

plt.suptitle('Colaterais e Garantias', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Países que mais deram colateral
col_paises = df[df['Collateralized/Securitized'] == 'Yes'].groupby('País')['AidData TUFF Project ID'].count().sort_values(ascending=False).head(12)

col_paises.sort_values().plot(kind='barh', color='tomato')
plt.title('Países com Mais Empréstimos Colateralizados')
plt.xlabel('Quantidade de Projetos')
plt.tight_layout()
plt.show()

---
## 5. Evolução Temporal dos Empréstimos

In [ ]:
# Valor total por ano (todos os projetos)
evolucao = df.groupby('Ano')['Valor_USD'].sum() / 1e9

evolucao.plot(kind='line', marker='o', color='steelblue', linewidth=2)
plt.title('Evolução dos Empréstimos Chineses ao Longo do Tempo')
plt.ylabel('Bilhões de USD (constante 2017)')
plt.xlabel('Ano')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# Evolução: BRI vs Não-BRI ao longo do tempo
evolucao_bri    = df[df['BRI-Like'] == 1].groupby('Ano')['Valor_USD'].sum() / 1e9
evolucao_naobri = df[df['Non-BRI'] == 1].groupby('Ano')['Valor_USD'].sum() / 1e9

plt.figure(figsize=(12, 5))
evolucao_bri.plot(label='BRI', marker='o', color='darkorange', linewidth=2)
evolucao_naobri.plot(label='Não-BRI', marker='s', color='steelblue', linewidth=2)
plt.title('Evolução Temporal: Empréstimos BRI vs Não-BRI')
plt.ylabel('Bilhões de USD (constante 2017)')
plt.xlabel('Ano')
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# Evolução por região
evolucao_regiao = (
    df.groupby(['Ano', 'Região'])['Valor_USD']
    .sum()
    .unstack()
    .fillna(0) / 1e9
)

evolucao_regiao.plot(kind='area', stacked=True, figsize=(12, 5), colormap='tab10', alpha=0.8)
plt.title('Empréstimos Chineses por Região ao Longo do Tempo')
plt.ylabel('Bilhões de USD (constante 2017)')
plt.xlabel('Ano')
plt.legend(loc='upper left', fontsize=9)
plt.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()